# 🎬 مولّد فيديوهات الأطفال 3D (مجاني، على Colab)
موديل مفتوح المصدر (Wan 2.1) بوضعين:
1. **نص ← فيديو:** تكتب المشاهد ويولّدها ويدمجها في فيديو عمودي (9:16).
2. **فيديو ← فيديو:** ترفع فيديو (مثلاً كرتون) ويحوّله إلى أسلوب 3D، مع إبقاء صوت الفيديو الأصلي.

**قبل البدء:** *Runtime* ← *Change runtime type* ← **T4 GPU**. شغّل الخلايا بالترتيب (1، 2، 3، 4)، ثم خلية الوضع الذي تريده (5 أو 6).

⚠️ الدقة الأصلية للموديل 480p. الخلية 2 تكبّر الناتج إلى 720×1280 وتحدّه (تحسين الوضوح، لا تفاصيل جديدة).

In [ ]:
# 1) التثبيت
!pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg ftfy

In [ ]:
# 2) الإعدادات المشتركة
QUALITY = "fast"   # "fast" سريع | "balanced" متوازن | "high" أعلى جودة (أبطأ بكثير)
PRESETS = {"fast": (33, 20), "balanced": (49, 30), "high": (81, 50)}   # (عدد الإطارات، الخطوات)
NUM_FRAMES, STEPS = PRESETS[QUALITY]
FPS = 16
WIDTH, HEIGHT = 480, 832          # عمودي 9:16 (للأفقي بدّل القيمتين)
UPSCALE_TO = (720, 1280)          # None لإلغاء التكبير
SHARPEN = True                    # زيادة الحدّة
INTERPOLATE_30FPS = False         # True = حركة أنعم (30fps) لكن أبطأ
NEGATIVE = "Bright tones, overexposed, static, blurred details, subtitles, worst quality, low quality, JPEG artifacts, deformed, extra limbs, ugly, watermark, text"
STYLE = "3D animated children's movie style, adorable big-eyed cartoon characters, vibrant colors, soft cinematic lighting, smooth motion, ultra detailed, sharp focus"

In [ ]:
# 3) تحميل الموديل (مرة واحدة) — بدون مشفّر النص لتفادي انهيار الذاكرة (RAM)
import torch, os, shutil, subprocess, gc
from diffusers import AutoencoderKLWan, WanPipeline, WanVideoToVideoPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video, load_video
from transformers import UMT5EncoderModel

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(model_id, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=torch.bfloat16)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)
pipe.enable_model_cpu_offload()
try:
    v2v = WanVideoToVideoPipeline.from_pipe(pipe)     # يشارك نفس الأجزاء (يوفّر الذاكرة)
except Exception:
    v2v = WanVideoToVideoPipeline(**pipe.components)

def encode_prompts(prompts):
    """يحمّل مشفّر النص مباشرة على الـ GPU (بلا استهلاك RAM)، يحسب التضمينات، ثم يحرّره."""
    te = UMT5EncoderModel.from_pretrained(model_id, subfolder="text_encoder",
                                          torch_dtype=torch.bfloat16, device_map={"": 0})
    tp = WanPipeline.from_pretrained(model_id, transformer=None, vae=None, text_encoder=te, torch_dtype=torch.bfloat16)
    out = []
    with torch.no_grad():
        for p in prompts:
            pe, ne = tp.encode_prompt(prompt=p, negative_prompt=NEGATIVE, do_classifier_free_guidance=True,
                                      max_sequence_length=512, device="cuda")
            out.append((pe.cpu(), ne.cpu()))
    del tp, te
    gc.collect(); torch.cuda.empty_cache()
    return out
print("جاهز")

In [ ]:
# 4) دالة الدمج والتحسين والتحميل
def finalize(clip_dir, out="final.mp4", audio_src=None):
    names = sorted(n for n in os.listdir(clip_dir) if n.endswith(".mp4") and n != "merged.mp4")
    with open(f"{clip_dir}/list.txt", "w") as f:
        for n in names: f.write(f"file '{n}'\n")
    vf = []
    if UPSCALE_TO: vf.append(f"scale={UPSCALE_TO[0]}:{UPSCALE_TO[1]}:flags=lanczos")
    if SHARPEN: vf.append("unsharp=5:5:0.8:5:5:0.0")
    if INTERPOLATE_30FPS: vf.append("minterpolate=fps=30:mi_mode=mci:mc_mode=aobmc:vsbmc=1")
    vf.append("format=yuv420p")
    merged = f"{clip_dir}/merged.mp4"
    subprocess.run(["ffmpeg","-y","-loglevel","error","-f","concat","-safe","0","-i",f"{clip_dir}/list.txt",
                    "-vf",",".join(vf),"-c:v","libx264","-crf","16","-preset","medium",merged], check=True)
    if audio_src:
        subprocess.run(["ffmpeg","-y","-loglevel","error","-i",merged,"-i",audio_src,"-map","0:v","-map","1:a?",
                        "-c:v","copy","-c:a","aac","-shortest",out], check=True)
    else:
        shutil.copy(merged, out)
    from google.colab import files
    files.download(out)
    return out
print("تمام")

## 5) نص ← فيديو
عدّل قائمة المشاهد ثم شغّل الخلية.

In [ ]:
SCENES = [
    "a cute baby with big blue eyes riding a friendly tiger on a sunny forest path with flowers and mountains",
    "a cute baby riding a smiling baby elephant through a colorful flower meadow, a wooden cottage in the background",
    "a cute baby riding a happy panda across a wooden bridge near a waterfall, pink blossom trees",
    "a cute baby riding a playful dolphin in sparkling blue water at sunset, splashing waves",
]
shutil.rmtree("clips", ignore_errors=True); os.makedirs("clips")
print("أحسب تضمين النصوص (دقيقة أو اثنتان) ...")
embs = encode_prompts([f"{s}. {STYLE}" for s in SCENES])
for i, (pe, ne) in enumerate(embs, 1):
    print(f"المقطع {i}/{len(SCENES)} ...")
    frames = pipe(prompt_embeds=pe.to("cuda"), negative_prompt_embeds=ne.to("cuda"), height=HEIGHT, width=WIDTH,
                  num_frames=NUM_FRAMES, num_inference_steps=STEPS, guidance_scale=5.0).frames[0]
    export_to_video(frames, f"clips/clip_{i:02d}.mp4", fps=FPS)
    torch.cuda.empty_cache()
finalize("clips")

## 6) فيديو ← فيديو (مثلاً كرتون ← 3D)
شغّل الخلية، ارفع الفيديو عند الطلب. `STRENGTH` كلما زاد تغيّر الشكل أكثر (0.6 يحافظ على الأصل، 0.85 يغيّر كثيراً).
الفيديو يُقسّم إلى مقاطع، وقد يظهر اختلاف بسيط في الشكل بين مقطع وآخر.

In [ ]:
from google.colab import files
V2V_PROMPT = "3D animated children's movie style, cute characters with big expressive eyes, vibrant colors, soft cinematic lighting, smooth motion, ultra detailed, sharp focus"
STRENGTH = 0.7
MAX_SECONDS = 12      # أقصى مدة تُعالَج من الفيديو

up = files.upload()
src = next(iter(up))
subprocess.run(["ffmpeg","-y","-loglevel","error","-i",src,"-t",str(MAX_SECONDS),
                "-vf",f"fps={FPS},scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=increase,crop={WIDTH}:{HEIGHT}",
                "-an","src16.mp4"], check=True)
frames = load_video("src16.mp4")
n = NUM_FRAMES
chunks = [frames[s:s+n] for s in range(0, len(frames) - n + 1, n)]
if not chunks and len(frames) >= 5:
    m = ((len(frames) - 1) // 4) * 4 + 1
    chunks = [frames[:m]]
shutil.rmtree("v2v", ignore_errors=True); os.makedirs("v2v")
print("أحسب تضمين النص ...")
(pe, ne), = encode_prompts([V2V_PROMPT])
for i, ch in enumerate(chunks, 1):
    print(f"المقطع {i}/{len(chunks)} ...")
    out = v2v(video=ch, prompt_embeds=pe.to("cuda"), negative_prompt_embeds=ne.to("cuda"), height=HEIGHT, width=WIDTH,
              strength=STRENGTH, guidance_scale=5.0, num_inference_steps=STEPS).frames[0]
    export_to_video(out, f"v2v/clip_{i:02d}.mp4", fps=FPS)
    torch.cuda.empty_cache()
finalize("v2v", audio_src=src)